# TrafficGuard · ResNet18 Congestion Classifier
**COMP47250 · Team Software Project · Project P14 · UCD Summer 2026**

**Owner:** Soham (Model Training) &nbsp;|&nbsp; **Contributor:** Yashi (FastAPI Backend)

---

### What this notebook does
Trains a ResNet18 image classifier to predict urban road congestion level —
**Low**, **Medium**, or **High** — from MIO-TCD traffic camera images.

Reads directly from **`labelled_manifest.csv`** produced by
`mio_tcd_pipeline_v2.ipynb` (Walid). No relabelling happens here.

---

### Notebook Structure

| # | Section | Description |
|---|---|---|
| 0 | Setup & Imports | Libraries and version checks |
| 1 | Configuration & Paths | All paths and hyperparameters in one place |
| 2 | Dataset & DataLoaders | Dataset class, transforms, loaders, class distribution |
| 3 | Model Architecture | ResNet18 definition and forward pass check |
| 4 | Training | Loss, optimiser, scheduler, training loop, curves |
| 5 | Evaluation | Test accuracy, classification report, confusion matrix |
| 6 | Save & Export | Checkpoint contents, loading template for teammates |
| 7 | Inference / Predict | Single-image prediction function (used by FastAPI) |
| 8 | Summary | Results, rubric check, next steps |

---

### Inputs & Outputs

| Direction | Path |
|---|---|
| **Input** | `BASE_PATH/pipeline_output/labelled_manifest.csv` (Walid's notebook output) |
| **Output** | `checkpoints/best.pt` · `checkpoints/last.pt` · `checkpoints/train_log.csv` |

---

### Label mapping (from pipeline notebook `assign_label()`)

| Integer index | String label | Rule from pipeline |
|---|---|---|
| 0 | Low | `vehicle_count < 3` → always Low; or `density_score ≤ p33` |
| 1 | Medium | `vehicle_count < 6` → always Medium; or `p33 < density_score ≤ p66` |
| 2 | High | `vehicle_count ≥ 6` and `density_score > p66` |

---
## 0. Setup & Imports

In [1]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import csv
import time
import random
from pathlib import Path

# ── Data ──────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from PIL import Image

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

# ── Torchvision ───────────────────────────────────────────────────────────────
from torchvision import transforms, models

# ── Metrics ───────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

print(f'PyTorch      : {torch.__version__}')
print(f'Torchvision  : {__import__("torchvision").__version__}')
print('All imports OK.')

ModuleNotFoundError: No module named 'numpy'

---
## 1. Configuration & Paths

All paths and hyperparameters are defined here. **Only edit this section.**

> **Cross-machine note:** The pipeline notebook (Walid) saves `image_path` as an
> absolute path on his machine (e.g. `C:\Users\riana\...`).
> If you are running on a different machine, set `IMAGES_DIR` to your local
> `MIO-TCD-Localization/train/` folder. The dataset class will resolve paths
> using `image_id` + `IMAGES_DIR` instead of the manifest column.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Root of the MIO-TCD-Localization folder on THIS machine
BASE_PATH = Path(r'C:\Users\Soham Patil\OneDrive\Desktop\trafficguard_p14\trafficguard-p14\data\raw\MIO-TCD-Localization')

# Manifest CSV produced by mio_tcd_pipeline_v2.ipynb (Walid)
MANIFEST_PATH = BASE_PATH / 'pipeline_output' / 'labelled_manifest.csv'

# Folder containing the raw .jpg images on THIS machine
# Set to None to use image_path column from manifest as-is
IMAGES_DIR = BASE_PATH / 'train'

# Where model checkpoints and plots will be saved
CHECKPOINT_DIR = Path('checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)

# ── Hyperparameters ───────────────────────────────────────────────────────────
EPOCHS        = 25
BATCH_SIZE    = 32    # reduce to 16 if you get a CUDA out-of-memory error
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-4
NUM_WORKERS   = 0     # keep 0 on Windows; set to 4 on Linux/Mac
RANDOM_SEED   = 42

# ── Label mapping — must match pipeline notebook assign_label() output ─────────
LABEL_MAP   = {'Low': 0, 'Medium': 1, 'High': 2}
CLASS_NAMES = ['Low', 'Medium', 'High']
NUM_CLASSES = len(CLASS_NAMES)

# Colour scheme (matches pipeline notebook)
CLASS_COLORS = {'Low': '#4CAF50', 'Medium': '#FF9800', 'High': '#F44336'}

# ── Reproducibility ───────────────────────────────────────────────────────────
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device         : {DEVICE}' +
      (f' ({torch.cuda.get_device_name(0)})' if DEVICE.type == 'cuda' else ''))
print(f'Manifest       : {MANIFEST_PATH}')
print(f'Images dir     : {IMAGES_DIR}')
print(f'Checkpoint dir : {CHECKPOINT_DIR}')
print(f'Epochs         : {EPOCHS}  |  Batch size: {BATCH_SIZE}  |  LR: {LEARNING_RATE}')

---
## 2. Dataset & DataLoaders

### 2.1 Image Transforms

Training images are augmented to improve generalisation.
Val and test images are **not** augmented — they must match inference-time preprocessing exactly.

| Transform | Train | Val / Test | Reason |
|---|---|---|---|
| Resize 256 → RandomCrop 224 | ✓ | ✗ | Positional variation |
| RandomHorizontalFlip | ✓ | ✗ | Roads look the same mirrored |
| ColorJitter | ✓ | ✗ | Lighting and weather changes |
| RandomRotation ±5° | ✓ | ✗ | Slight camera tilt |
| Resize directly to 224 | ✗ | ✓ | Deterministic, matches inference |
| ToTensor + ImageNet Normalize | ✓ | ✓ | Required for pretrained ResNet18 |

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def get_transforms(split: str) -> transforms.Compose:
    """
    Return the appropriate transform pipeline for each data split.

    Args:
        split : 'train' | 'val' | 'test'
    """
    if split == 'train':
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomCrop(224),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(
                brightness=0.3, contrast=0.3, saturation=0.15, hue=0.05
            ),
            transforms.RandomRotation(degrees=5),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    else:
        # Val and test: deterministic, no augmentation
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])


print('Transforms defined.')
print(f'  Train : {len(get_transforms("train").transforms)} steps (augmented)')
print(f'  Val   : {len(get_transforms("val").transforms)} steps (deterministic)')
print(f'  Test  : {len(get_transforms("test").transforms)} steps (deterministic)')

### 2.2 Dataset Class

Reads the `labelled_manifest.csv` from Walid's pipeline notebook.
Handles cross-machine path resolution via `IMAGES_DIR`.

In [ ]:
class MIOTCDDataset(Dataset):
    """
    PyTorch Dataset for the MIO-TCD congestion manifest.

    Reads labelled_manifest.csv produced by mio_tcd_pipeline_v2.ipynb.

    Expected manifest columns:
        image_id | image_path | vehicle_count | hull_area | hull_coverage |
        packing_density | mean_nn_dist_norm | density_score |
        congestion_label | split

    Args:
        manifest_path : path to labelled_manifest.csv
        split         : 'train' | 'val' | 'test'
        transform     : torchvision transform pipeline
                        (defaults to get_transforms(split) if None)
        images_dir    : if provided, image paths are resolved as
                        images_dir / image_id.jpg, overriding the
                        image_path column in the manifest.
                        Use this when running on a different machine
                        to whoever generated the manifest.
    """

    def __init__(self, manifest_path, split, transform=None, images_dir=None):
        df = pd.read_csv(manifest_path, dtype={'image_id': str})

        # ── Validate required columns ──────────────────────────────────────────
        required = {'image_id', 'congestion_label', 'split'}
        missing  = required - set(df.columns)
        if missing:
            raise ValueError(
                f'Manifest is missing columns: {missing}\n'
                f'Re-run mio_tcd_pipeline_v2.ipynb to regenerate it.'
            )

        # ── Filter to requested split ──────────────────────────────────────────
        self.df         = df[df['split'] == split].reset_index(drop=True)
        self.transform  = transform or get_transforms(split)
        self.images_dir = Path(images_dir) if images_dir else None

        # ── Validate label values ──────────────────────────────────────────────
        unknown = set(self.df['congestion_label'].unique()) - set(LABEL_MAP)
        if unknown:
            raise ValueError(f'Unknown congestion_label values: {unknown}')

        # Map string labels → integer indices
        self.df = self.df.copy()
        self.df['label_idx'] = self.df['congestion_label'].map(LABEL_MAP)

        # ── Class counts (for distribution plot and weighted loss) ─────────────
        self._counts = self.df['congestion_label'].value_counts()
        total = len(self.df)
        self.class_weights = torch.tensor(
            [total / (NUM_CLASSES * self._counts.get(c, 1)) for c in CLASS_NAMES],
            dtype=torch.float32,
        )

    def __len__(self):
        return len(self.df)

    def _resolve_path(self, row):
        """Resolve image file path with optional root override."""
        if self.images_dir:
            return self.images_dir / f"{row['image_id']}.jpg"
        return Path(row['image_path'])

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = self._resolve_path(row)
        label    = int(row['label_idx'])

        try:
            image = Image.open(img_path).convert('RGB')
        except (FileNotFoundError, OSError) as e:
            raise FileNotFoundError(
                f'Image not found: {img_path}\n'
                f'Tip: set IMAGES_DIR to your local MIO-TCD-Localization/train/ folder.'
            ) from e

        if self.transform:
            image = self.transform(image)

        return image, label


print('MIOTCDDataset defined.')

### 2.3 Load Datasets & Inspect Class Distribution

In [ ]:
train_dataset = MIOTCDDataset(MANIFEST_PATH, 'train', images_dir=IMAGES_DIR)
val_dataset   = MIOTCDDataset(MANIFEST_PATH, 'val',   images_dir=IMAGES_DIR)
test_dataset  = MIOTCDDataset(MANIFEST_PATH, 'test',  images_dir=IMAGES_DIR)

print(f'Dataset sizes:')
print(f'  Train : {len(train_dataset):>6,}')
print(f'  Val   : {len(val_dataset):>6,}')
print(f'  Test  : {len(test_dataset):>6,}')
print(f'  Total : {len(train_dataset)+len(val_dataset)+len(test_dataset):>6,}')

In [ ]:
# ── Class distribution bar chart ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
split_names  = ['Train', 'Val', 'Test']
split_datasets = [train_dataset, val_dataset, test_dataset]

for ax, ds, name in zip(axes, split_datasets, split_names):
    counts = [int((ds.df['congestion_label'] == c).sum()) for c in CLASS_NAMES]
    bars   = ax.bar(
        CLASS_NAMES, counts,
        color=[CLASS_COLORS[c] for c in CLASS_NAMES],
        edgecolor='white', width=0.5
    )
    for bar, val in zip(bars, counts):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(counts) * 0.01,
            f'{val:,}', ha='center', va='bottom', fontsize=10
        )
    ax.set_title(f'{name} ({sum(counts):,} images)')
    ax.set_xlabel('Congestion class')
    ax.set_ylabel('Images')
    ax.set_ylim(0, max(counts) * 1.15)

fig.suptitle('Class distribution across splits  (from pipeline manifest)', fontsize=12)
plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / 'class_distribution.png', dpi=120)
plt.show()

print('Class weights (inverse frequency — use if distribution is skewed):')
for cls, w in zip(CLASS_NAMES, train_dataset.class_weights.tolist()):
    print(f'  {cls:<8}: {w:.4f}')

### 2.4 DataLoaders

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == 'cuda'),
    drop_last=True,    # avoids BatchNorm errors on an uneven final batch
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == 'cuda'),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == 'cuda'),
)

print(f'Train batches : {len(train_loader):,}')
print(f'Val batches   : {len(val_loader):,}')
print(f'Test batches  : {len(test_loader):,}')

# ── Quick shape check ──────────────────────────────────────────────────────────
sample_imgs, sample_labels = next(iter(train_loader))
print(f'\nBatch shape  : {tuple(sample_imgs.shape)}  (batch, channels, H, W)')
print(f'Label values : {sample_labels.unique().tolist()}  (should be subset of [0, 1, 2])')

### 2.5 Visual Sample Check

Confirm images and labels look correct before training.

In [ ]:
def denormalise(tensor):
    """Reverse ImageNet normalisation for display."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)


n_show = min(8, BATCH_SIZE)
fig, axes = plt.subplots(1, n_show, figsize=(2.5 * n_show, 3))

for i, ax in enumerate(axes):
    img   = denormalise(sample_imgs[i]).permute(1, 2, 0).numpy()
    label = CLASS_NAMES[sample_labels[i].item()]
    ax.imshow(img)
    ax.set_title(label, fontsize=9, color=CLASS_COLORS[label], fontweight='bold')
    ax.axis('off')

fig.suptitle('Sample training images after augmentation', fontsize=11)
plt.tight_layout()
plt.show()

---
## 3. Model Architecture

### 3.1 Why ResNet18?

| Model | Expected accuracy | Training time | ART support | Decision |
|---|---|---|---|---|
| ResNet18 | ≥ 80% | ~30 min / 25 epochs | ✓ Full | **Selected** |
| ResNet50 | ~82% | ~90 min | ✓ Full | Overkill for 3 classes |
| ViT-B/16 | ~85% | 3–4 hrs | Partial | Too slow for 10-day timeline |

ResNet18 pretrained on ImageNet already understands edges, textures, and shapes.
Fine-tuning replaces only the final classification head.

### 3.2 Architecture change

```
Original ResNet18 final layer:  Linear(512 → 1000)   ← ImageNet
TrafficGuard final layer:       Dropout(0.3) → Linear(512 → 3)
```

Dropout(0.3) reduces overfitting on our relabelled dataset.

In [ ]:
def build_model(freeze_backbone: bool = False) -> nn.Module:
    """
    Load pretrained ResNet18 and replace the classifier head for
    3-class traffic congestion classification.

    Args:
        freeze_backbone : False (default) = full fine-tuning of all layers.
                          True = only the new FC head is trained.
                          Use True only for very quick prototyping.

    Returns:
        nn.Module ready to move to device and train.
    """
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # Replace final FC: Linear(512 → 1000) becomes Dropout + Linear(512 → 3)
    in_features = model.fc.in_features    # 512 for ResNet18
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, NUM_CLASSES),
    )

    # Always ensure the new head is trainable
    for param in model.fc.parameters():
        param.requires_grad = True

    return model


print('build_model() defined.')

### 3.3 Instantiate & Inspect

In [ ]:
model = build_model(freeze_backbone=False).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Architecture     : ResNet18 (ImageNet pretrained, full fine-tune)')
print(f'Total params     : {total_params:,}')
print(f'Trainable params : {trainable_params:,}')
print(f'Device           : {DEVICE}')
print(f'\nModified final layer:')
print(model.fc)

In [ ]:
# ── Forward pass sanity check ──────────────────────────────────────────────────
with torch.no_grad():
    dummy_input  = torch.randn(2, 3, 224, 224).to(DEVICE)
    dummy_output = model(dummy_input)
    dummy_probs  = F.softmax(dummy_output, dim=1)

print(f'Input shape  : {tuple(dummy_input.shape)}')
print(f'Output shape : {tuple(dummy_output.shape)}  (should be [2, 3])')
print(f'Probabilities: {dummy_probs.cpu().numpy().round(3)}')
print(f'Row sums     : {dummy_probs.sum(dim=1).cpu().numpy().round(4)}  (should be ~1.0)')
print(f'\n{"Forward pass OK" if dummy_output.shape == (2, 3) else "Shape mismatch — check NUM_CLASSES"}'  )

---
## 4. Training

### 4.1 Loss, Optimiser & Scheduler

| Component | Choice | Reason |
|---|---|---|
| Loss | CrossEntropyLoss | Standard for multi-class classification |
| Optimiser | AdamW | Better weight-decay handling than Adam for fine-tuning |
| Scheduler | CosineAnnealingLR | Smooth LR decay, avoids aggressive drops |

Set `USE_CLASS_WEIGHTS = True` if the class distribution from Section 2.3
shows one class with more than 2× the samples of another.

In [ ]:
# Set True if class imbalance is significant (check Section 2.3 plot)
USE_CLASS_WEIGHTS = False

if USE_CLASS_WEIGHTS:
    weights   = train_dataset.class_weights.to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)
    print(f'Loss: CrossEntropyLoss (weighted: {[round(w,3) for w in weights.tolist()]})')
else:
    criterion = nn.CrossEntropyLoss()
    print('Loss: CrossEntropyLoss (unweighted)')

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

print(f'Optimiser: AdamW  (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})')
print(f'Scheduler: CosineAnnealingLR  (T_max={EPOCHS}, eta_min=1e-6)')

### 4.2 Train and Evaluate Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """
    One full pass over the training set.
    Returns (avg_loss, accuracy).
    """
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds         = outputs.argmax(dim=1)
        correct      += (preds == labels).sum().item()
        total        += labels.size(0)

    return running_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    """
    Evaluate model on val or test loader.
    Returns (avg_loss, accuracy, all_preds, all_labels).
    """
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels        = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss    = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds         = outputs.argmax(dim=1)
            correct      += (preds == labels).sum().item()
            total        += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return running_loss / total, correct / total, all_preds, all_labels


print('train_one_epoch() and evaluate() defined.')

### 4.3 Run Training

Best checkpoint (highest val accuracy) saved automatically to `checkpoints/best.pt`.
All per-epoch metrics logged to `checkpoints/train_log.csv`.

In [ ]:
best_val_acc = 0.0
log_path     = CHECKPOINT_DIR / 'train_log.csv'

# Write CSV header
with open(log_path, 'w', newline='') as f:
    csv.writer(f).writerow(
        ['epoch', 'train_loss', 'train_acc', 'val_loss', 'val_acc', 'lr']
    )

header = (f"{'Epoch':>6}  {'Train Loss':>10}  {'Train Acc':>9}  "
          f"{'Val Loss':>9}  {'Val Acc':>8}  {'LR':>9}  {'Time':>7}")
print(header)
print('─' * len(header))

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    train_loss, train_acc   = train_one_epoch(
        model, train_loader, criterion, optimizer, DEVICE
    )
    val_loss, val_acc, _, _ = evaluate(
        model, val_loader, criterion, DEVICE
    )
    scheduler.step()

    current_lr = scheduler.get_last_lr()[0]
    elapsed    = time.time() - t0

    # Log to CSV
    with open(log_path, 'a', newline='') as f:
        csv.writer(f).writerow(
            [epoch, train_loss, train_acc, val_loss, val_acc, current_lr]
        )

    saved = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(
            {
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc':              val_acc,
                'class_names':          CLASS_NAMES,
                'label_map':            LABEL_MAP,
            },
            CHECKPOINT_DIR / 'best.pt',
        )
        saved = '  ← best saved ✓'

    print(
        f"{epoch:>6}  {train_loss:>10.4f}  {train_acc:>8.2%}  "
        f"{val_loss:>9.4f}  {val_acc:>7.2%}  "
        f"{current_lr:>9.2e}  {elapsed:>6.1f}s" + saved
    )

# Save final epoch checkpoint
torch.save(
    {'epoch': EPOCHS, 'model_state_dict': model.state_dict(), 'class_names': CLASS_NAMES},
    CHECKPOINT_DIR / 'last.pt',
)

print(f'\nTraining complete.')
print(f'Best val accuracy : {best_val_acc:.2%}')
print(f'Checkpoints saved to : {CHECKPOINT_DIR}/')

### 4.4 Training Curves

In [ ]:
log_df = pd.read_csv(log_path)
epochs = log_df['epoch'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss
axes[0].plot(epochs, log_df['train_loss'], label='Train loss',
             color='steelblue', linewidth=2)
axes[0].plot(epochs, log_df['val_loss'],   label='Val loss',
             color='darkorange', linewidth=2, linestyle='--')
axes[0].set_title('Loss per epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs, log_df['train_acc'], label='Train acc',
             color='steelblue', linewidth=2)
axes[1].plot(epochs, log_df['val_acc'],   label='Val acc',
             color='darkorange', linewidth=2, linestyle='--')
axes[1].axhline(0.80, color='green', linestyle=':', linewidth=1.5,
                label='Rubric target (80%)')
axes[1].set_title('Accuracy per epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1.05)
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.suptitle('TrafficGuard ResNet18 · Training Curves', fontsize=13)
plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / 'training_curves.png', dpi=120)
plt.show()
print('Saved: training_curves.png')

---
## 5. Evaluation on Test Set

Load the **best checkpoint** (not last epoch) and evaluate on the held-out test set.

### 5.1 Load Best Checkpoint

In [ ]:
best_model = build_model().to(DEVICE)
ckpt       = torch.load(CHECKPOINT_DIR / 'best.pt', map_location=DEVICE)
best_model.load_state_dict(ckpt['model_state_dict'])
best_model.eval()

print(f"Loaded checkpoint  : epoch {ckpt['epoch']}, val_acc = {ckpt['val_acc']:.2%}")
print('Model in eval mode : ready for test evaluation.')

### 5.2 Test Accuracy & Classification Report

In [ ]:
_, test_acc, test_preds, test_labels = evaluate(
    best_model, test_loader, criterion, DEVICE
)

print('=' * 55)
print(f'  Clean Test Accuracy : {test_acc:.2%}')
print('=' * 55)
print()
print('Classification Report:')
print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES))

if test_acc >= 0.80:
    print('✓ Meets rubric minimum (≥ 80% clean accuracy)')
else:
    print(f'✗ Below rubric minimum by {0.80 - test_acc:.2%}')
    print('  Try: more epochs · lower LR · USE_CLASS_WEIGHTS = True')

### 5.3 Confusion Matrix

In [ ]:
cm  = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(6, 5))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
ax.set_title(f'Confusion Matrix · Test set  (acc = {test_acc:.2%})', fontsize=12)
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')

plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / 'confusion_matrix.png', dpi=120)
plt.show()
print('Saved: confusion_matrix.png')

### 5.4 Per-Class Accuracy

In [ ]:
per_class_acc = cm.diagonal() / cm.sum(axis=1)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    CLASS_NAMES, per_class_acc,
    color=[CLASS_COLORS[c] for c in CLASS_NAMES],
    edgecolor='white', width=0.5
)
ax.axhline(0.80, color='gray', linestyle='--', linewidth=1.2, label='80% target')
for bar, val in zip(bars, per_class_acc):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f'{val:.1%}', ha='center', va='bottom', fontsize=11
    )
ax.set_ylim(0, 1.12)
ax.set_title('Per-class accuracy on test set')
ax.set_xlabel('Congestion class')
ax.set_ylabel('Accuracy')
ax.legend()
plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / 'per_class_accuracy.png', dpi=120)
plt.show()
print('Saved: per_class_accuracy.png')

---
## 6. Save & Export

### 6.1 Checkpoint Contents

`checkpoints/best.pt` contains everything Payal (FGSM) and Rian (defence)
need to load the model — including `class_names` and `label_map`
so they don't need to hardcode them.

In [ ]:
ckpt_inspect = torch.load(CHECKPOINT_DIR / 'best.pt', map_location='cpu')

print('Contents of best.pt:')
for key, val in ckpt_inspect.items():
    if key == 'model_state_dict':
        print(f'  {key:<30} : dict with {len(val)} tensors')
    elif key == 'optimizer_state_dict':
        print(f'  {key:<30} : dict (optimiser state)')
    else:
        print(f'  {key:<30} : {val}')

### 6.2 Loading Template for Payal (attacks) and Rian (defence)

Copy this function into `attacks/fgsm.ipynb` and `defence/smoothing.ipynb`.
It returns the model in `eval()` mode, ready to wrap with ART's `PyTorchClassifier`.

In [ ]:
def load_trained_model(checkpoint_path, device=None):
    """
    Load a saved TrafficGuard ResNet18 checkpoint.

    Returns the model in eval() mode — ready for ART wrapping.

    Usage (in Payal's or Rian's notebook):
        model = load_trained_model('checkpoints/best.pt')

    Then wrap with ART:
        from art.estimators.classification import PyTorchClassifier
        classifier = PyTorchClassifier(
            model=model,
            loss=nn.CrossEntropyLoss(),
            input_shape=(3, 224, 224),
            nb_classes=3,
        )
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    m    = build_model().to(device)
    ckpt = torch.load(checkpoint_path, map_location=device)

    # Support both full checkpoint dict and raw state_dict
    state_dict = ckpt.get('model_state_dict', ckpt)
    m.load_state_dict(state_dict)
    m.eval()

    print(f"Loaded checkpoint : epoch {ckpt.get('epoch','?')}, "
          f"val_acc = {ckpt.get('val_acc', 0):.2%}")
    print(f'Device            : {device}')
    return m


# Quick test
reload_test = load_trained_model(CHECKPOINT_DIR / 'best.pt', device=DEVICE)
print('\n✓ load_trained_model() works correctly.')

---
## 7. Inference / Predict

This is the same prediction flow Yashi will use in the FastAPI backend.
Given a single image (PIL, file path, or numpy array), return the
predicted congestion label and confidence.

### 7.1 Inference Transform

Must be **identical** to the val/test transform — no augmentation.

In [ ]:
INFERENCE_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

print('Inference transform defined (matches val/test — no augmentation).')

### 7.2 Predict Functions

In [ ]:
def predict_pil(model, image: Image.Image, device) -> dict:
    """
    Run inference on a PIL image.

    Args:
        model  : loaded ResNet18 in eval() mode
        image  : PIL.Image.Image (RGB)
        device : torch.device

    Returns:
        {
          'label':         str   — 'Low' | 'Medium' | 'High'
          'label_idx':     int   — 0 | 1 | 2
          'confidence':    float — max softmax probability
          'probabilities': dict  — {'Low': float, 'Medium': float, 'High': float}
        }
    """
    tensor = INFERENCE_TRANSFORM(image).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(tensor)
        probs  = F.softmax(logits, dim=1).squeeze(0).cpu().numpy()

    label_idx = int(probs.argmax())
    return {
        'label':         CLASS_NAMES[label_idx],
        'label_idx':     label_idx,
        'confidence':    round(float(probs[label_idx]), 4),
        'probabilities': {cls: round(float(probs[i]), 4)
                          for i, cls in enumerate(CLASS_NAMES)},
    }


def predict_path(model, image_path: str, device) -> dict:
    """Convenience wrapper: load image from disk then call predict_pil."""
    image = Image.open(image_path).convert('RGB')
    return predict_pil(model, image, device)


def predict_numpy(model, array: np.ndarray, device) -> dict:
    """
    Run inference on a numpy array (H, W, C) uint8 [0–255].
    Used by OpenCV video frames from the live dashboard stream.
    """
    image = Image.fromarray(array.astype(np.uint8)).convert('RGB')
    return predict_pil(model, image, device)


print('predict_pil(), predict_path(), predict_numpy() defined.')

### 7.3 Single Image Inference Demo

In [ ]:
# Pick a random image from the test set and run full inference
sample_row  = test_dataset.df.sample(1, random_state=99).iloc[0]
sample_path = test_dataset._resolve_path(sample_row)
true_label  = sample_row['congestion_label']

result = predict_path(best_model, sample_path, DEVICE)

# ── Display ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Image
img = Image.open(sample_path).convert('RGB')
axes[0].imshow(img)
axes[0].axis('off')
match = '✓ Correct' if result['label'] == true_label else '✗ Wrong'
axes[0].set_title(
    f"{match}\nTrue: {true_label}  |  Predicted: {result['label']} ({result['confidence']:.1%})",
    fontsize=10
)

# Probability bar chart
probs  = [result['probabilities'][c] for c in CLASS_NAMES]
bcolors = [CLASS_COLORS[c] for c in CLASS_NAMES]
axes[1].barh(CLASS_NAMES, probs, color=bcolors, edgecolor='white')
axes[1].set_xlim(0, 1)
axes[1].set_xlabel('Probability')
axes[1].set_title('Prediction confidence breakdown')
for i, (p, name) in enumerate(zip(probs, CLASS_NAMES)):
    axes[1].text(p + 0.01, i, f'{p:.1%}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print('\nFull result dict:')
for k, v in result.items():
    print(f'  {k:<15} : {v}')

### 7.4 Batch Inference on Multiple Test Images

In [ ]:
# Run inference on 12 test images and display a grid
n_samples = 12
sample_rows = test_dataset.df.sample(n_samples, random_state=7).reset_index(drop=True)

fig, axes = plt.subplots(3, 4, figsize=(14, 9))
axes = axes.flatten()

for i, (_, row) in enumerate(sample_rows.iterrows()):
    path    = test_dataset._resolve_path(row)
    true    = row['congestion_label']
    result  = predict_path(best_model, path, DEVICE)
    pred    = result['label']
    conf    = result['confidence']
    correct = pred == true

    img = Image.open(path).convert('RGB')
    axes[i].imshow(img)
    axes[i].axis('off')
    colour = CLASS_COLORS[pred]
    marker = '✓' if correct else '✗'
    axes[i].set_title(
        f'{marker} T:{true}  P:{pred}\n({conf:.0%})',
        fontsize=8, color=colour if correct else 'red'
    )

fig.suptitle('Batch inference on test images  (T=True, P=Predicted)', fontsize=12)
plt.tight_layout()
plt.show()

---
## 8. Summary

In [ ]:
print('TRAFFICGUARD — MODEL NOTEBOOK SUMMARY')
print('=' * 58)
print(f'Model            : ResNet18 (ImageNet pretrained, full fine-tune)')
print(f'Trainable params : {trainable_params:,}')
print(f'Dataset          : MIO-TCD Localization (via labelled_manifest.csv)')
print(f'Classes          : {CLASS_NAMES}')
print()
print('Training config:')
print(f'  Epochs           : {EPOCHS}')
print(f'  Batch size       : {BATCH_SIZE}')
print(f'  Learning rate    : {LEARNING_RATE}')
print(f'  Optimiser        : AdamW (weight_decay={WEIGHT_DECAY})')
print(f'  Scheduler        : CosineAnnealingLR')
print(f'  Class weights    : {USE_CLASS_WEIGHTS}')
print()
print('Results:')
print(f'  Best val acc     : {best_val_acc:.2%}')
print(f'  Test accuracy    : {test_acc:.2%}')
print(f'  Rubric target    : {"MET ✓" if test_acc >= 0.80 else "NOT MET ✗"} (>= 80%)')
print()
print('Saved files:')
for fname in [
    'best.pt', 'last.pt', 'train_log.csv',
    'class_distribution.png', 'training_curves.png',
    'confusion_matrix.png', 'per_class_accuracy.png'
]:
    path   = CHECKPOINT_DIR / fname
    status = '✓' if path.exists() else '✗ missing'
    print(f'  {status}  {path}')
print()
print('Next steps:')
print('  Payal → load checkpoints/best.pt in attacks/fgsm.ipynb')
print('  Rian  → load checkpoints/best.pt in defence/smoothing.ipynb')
print('  Yashi → load checkpoints/best.pt in backend/main.py')